# 04 — JAX: CPU first, then TPU-ready

**Learning goal.** Walk through the smallest JAX-on-CPU example, then show the exact code you'd add to make the same notebook **TPU-ready** without breaking the CPU path.

**What you'll observe.**
- A `jax.jit`-compiled matmul running on CPU.
- A guarded "TPU-ready" cell that lists devices, places arrays, and re-jits — but only if JAX is installed (and just prints a notice otherwise).

**Knobs to tune.**
- `N` — matrix size.
- `dtype` — `bfloat16` vs `float32`.
- Inside a Cloud TPU VM, `JAX_PLATFORMS=tpu` selects the TPU backend.

**Failure modes to watch.**
- JAX may not be installed in your local kernel. The TPU-ready cell is guarded.
- `jax.jit` traces on first call — first-call time is mostly compile, not compute.
- Don't time the first call; warm up first, then time subsequent calls.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## CPU JAX: jit-compiled matmul

Falls back gracefully if JAX is not installed.

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    HAVE_JAX = True
except Exception as e:
    HAVE_JAX = False
    print("JAX is not installed in this kernel:", e)
    print("Skipping JAX cells. Install `jax` to run them locally on CPU,")
    print("or run this notebook inside a Cloud TPU VM.")

In [ ]:
if HAVE_JAX:
    print("jax version:", jax.__version__)
    print("devices:", jax.devices())
    print("default backend:", jax.default_backend())
else:
    print("(skipped — no JAX)")

In [ ]:
if HAVE_JAX:
    import time

    @jax.jit
    def matmul(a, b):
        return a @ b

    N = 256
    a = jnp.ones((N, N), dtype=jnp.float32)
    b = jnp.ones((N, N), dtype=jnp.float32)

    # Warm up the compile.
    _ = matmul(a, b).block_until_ready()

    t0 = time.perf_counter()
    for _ in range(10):
        c = matmul(a, b).block_until_ready()
    elapsed = (time.perf_counter() - t0) / 10
    print(f"jit'd matmul N={N}  avg {elapsed*1000:.2f} ms/call on {jax.default_backend()}")
else:
    print("(skipped — no JAX)")

## TPU-ready (only runs on Cloud TPU)

The block below is the **same JAX code** you'd use locally, but with two TPU idioms made explicit:

- `jax.devices()` returns TPU devices when the TPU runtime is available.
- `jax.device_put(x, device)` explicitly places an array on a device.

On CPU this still works — the device is the CPU. On a TPU VM it auto-uses the TPU runtime. We guard the whole thing so the notebook is safe to run anywhere.

In [ ]:
# TPU-ready (only runs on Cloud TPU; CPU-safe fallback below)
if HAVE_JAX:
    devices = jax.devices()
    is_tpu = any(getattr(d, "platform", "") == "tpu" for d in devices)
    print("detected devices:", devices)
    print("is_tpu:", is_tpu)

    target = devices[0]
    x = jnp.arange(16, dtype=jnp.float32).reshape(4, 4)
    x_dev = jax.device_put(x, target)
    print("placed on:", x_dev.device)

    @jax.jit
    def square_sum(t):
        return jnp.sum(t * t)

    out = square_sum(x_dev).block_until_ready()
    print("square_sum:", float(out))

    if not is_tpu:
        print("(running on CPU/GPU — on a real TPU VM the same code would target TPU cores)")
else:
    print("(skipped — no JAX)")

## What changes on a real Cloud TPU VM?

The code above is what you'd actually run. On the TPU VM:

- `jax.devices()` returns a list of `TpuDevice` objects.
- The first jit-compile of each unique shape pays an XLA compile cost (the same cache from notebook 03 lives inside JAX).
- For multi-host slices, you'd use `jax.distributed.initialize()` and then `pmap` / `shard_map` to fan out across devices.

For a deeper sharding walkthrough see notebook 07. For PJRT-level placement see notebook 05.

## Exercises

1. Time `matmul` for `N` in `[64, 256, 1024]`. Plot ms vs `N`. Is the slope cubic or sub-cubic?
2. Replace `float32` with `bfloat16` (or `jnp.bfloat16`). Does the CPU time change? On TPU it should — explain why.
3. Use `jax.make_jaxpr(matmul)(a, b)` to print the traced IR. Identify the dot_general op.
4. Try `jax.jit(matmul, static_argnums=...)` and pass shape as a Python int. What happens to recompiles?
5. From a Cloud TPU VM, run `jax.devices()` and report how many TpuDevices you see. Match against the chip count you provisioned.